# 02 — Generate EGFR mutant structures

This notebook automatically:

1. uploads the experimental `assay_outcomes.csv`,
2. downloads one EGFR–drug crystal structure for each TKI,
3. checks whether each mutation is actually present in that crystal structure,
4. introduces supported substitutions with **PDBFixer**,
5. preserves the original ligand coordinates,
6. saves one PDB file per supported mutation–drug pair, and
7. creates a report listing supported and unsupported mutations.

## Important scientific limitation

The drug-bound structures used here contain the **EGFR kinase domain**, not full-length EGFR. Therefore, mutations outside the crystallized residue range cannot be generated from these structures and will be reported as unsupported.

This notebook generates starting mutant coordinates. It does **not** yet perform ligand-aware energy minimization. That will be handled in the next notebook because covalent inhibitors and nonstandard ligands require careful parameterization.

## Step 1 — Upload `assay_outcomes.csv`

Use the file downloaded from Step 5 of the first notebook.

In [ ]:
from google.colab import files

uploaded = files.upload()
print("Uploaded:", list(uploaded))

## Step 2 — Install required packages

In [ ]:
!pip -q install pdbfixer openmm biopython pandas requests

## Step 3 — Load the mutation–drug table

This notebook expects these columns:

- `mutation`
- `drug`
- `auc_ratio_vs_wt`
- `resistant`


In [ ]:
import pandas as pd

assay = pd.read_csv("assay_outcomes.csv")

required = {"mutation", "drug", "auc_ratio_vs_wt", "resistant"}
missing = required.difference(assay.columns)
if missing:
    raise ValueError(f"assay_outcomes.csv is missing columns: {sorted(missing)}")

pairs = (
    assay[["mutation", "drug", "auc_ratio_vs_wt", "resistant"]]
    .drop_duplicates()
    .sort_values(["drug", "mutation"])
    .reset_index(drop=True)
)

print("Mutation–drug pairs:", len(pairs))
print("Distinct mutations:", pairs["mutation"].nunique())
print("Drugs:", sorted(pairs["drug"].unique()))
display(pairs.head(10))

## Step 4 — Define the drug-bound reference structures

These are editable. The current pilot mapping uses wild-type EGFR kinase structures where available:

- Erlotinib: **1M17**
- Afatinib: **4G5J**
- Dacomitinib: **4I23**
- Osimertinib/AZD9291: **4ZAU**


In [ ]:
DRUG_TO_PDB = {
    "Erlotinib": "1M17",
    "Afatinib": "4G5J",
    "Dacomitinib": "4I23",
    "Osimertinib": "4ZAU",
}

DRUG_TO_PDB

## Step 5 — Download the PDB structures

In [ ]:
from pathlib import Path
import requests

structure_dir = Path("reference_structures")
structure_dir.mkdir(exist_ok=True)

def download_pdb(pdb_id: str) -> Path:
    pdb_id = pdb_id.upper()
    out = structure_dir / f"{pdb_id}.pdb"
    if out.exists() and out.stat().st_size > 0:
        return out

    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    out.write_bytes(response.content)
    return out

reference_files = {}
for drug, pdb_id in DRUG_TO_PDB.items():
    path = download_pdb(pdb_id)
    reference_files[drug] = path
    print(f"{drug:12s} -> {pdb_id} -> {path}")

## Step 6 — Parse mutation notation and inspect residue coverage

Only simple substitutions such as `T790M` are supported in this notebook.

In [ ]:
import re
from Bio.PDB import PDBParser
from Bio.Data.IUPACData import protein_letters_3to1

THREE_TO_ONE = {k.upper(): v.upper() for k, v in protein_letters_3to1.items()}

def parse_mutation(text: str):
    match = re.fullmatch(r"([A-Z])(\d+)([A-Z])", str(text).strip().upper())
    if not match:
        raise ValueError(f"Unsupported mutation notation: {text}")
    wt, position, mutant = match.groups()
    return wt, int(position), mutant

def protein_residue_index(pdb_path):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("egfr", str(pdb_path))
    index = {}

    for model in structure:
        for chain in model:
            for residue in chain:
                hetflag, residue_number, insertion_code = residue.id
                if hetflag.strip():
                    continue
                resname = residue.get_resname().upper()
                one_letter = THREE_TO_ONE.get(resname)
                if one_letter:
                    index.setdefault(residue_number, []).append(
                        {
                            "chain": chain.id,
                            "one_letter": one_letter,
                            "resname": resname,
                            "insertion_code": insertion_code.strip(),
                        }
                    )
        break
    return index

coverage_rows = []
reference_indexes = {}

for drug, pdb_path in reference_files.items():
    idx = protein_residue_index(pdb_path)
    reference_indexes[drug] = idx
    positions = sorted(idx)
    coverage_rows.append({
        "drug": drug,
        "pdb_id": DRUG_TO_PDB[drug],
        "minimum_residue_number": min(positions),
        "maximum_residue_number": max(positions),
        "numbered_positions_present": len(positions),
    })

coverage = pd.DataFrame(coverage_rows)
display(coverage)

## Step 7 — Determine which mutation–drug pairs can be modeled

A pair is supported only when:

1. the residue number exists in the corresponding PDB,
2. the original amino acid in the PDB matches the mutation notation, and
3. there is an unambiguous protein chain containing that residue.

Unsupported cases are retained in the report rather than silently discarded.

In [ ]:
def locate_mutation(drug: str, mutation: str):
    try:
        wt, position, mutant = parse_mutation(mutation)
    except ValueError as exc:
        return {
            "supported": False,
            "reason": str(exc),
            "wt": None,
            "position": None,
            "mutant": None,
            "chain": None,
        }

    candidates = reference_indexes[drug].get(position, [])
    matching = [x for x in candidates if x["one_letter"] == wt]

    if not candidates:
        reason = "residue number absent from kinase-domain PDB"
    elif not matching:
        observed = ",".join(
            f'{x["chain"]}:{x["one_letter"]}{position}' for x in candidates
        )
        reason = f"WT residue mismatch; structure contains {observed}"
    elif len(matching) > 1:
        chains = ",".join(x["chain"] for x in matching)
        reason = f"ambiguous: matching residue occurs in multiple chains ({chains})"
    else:
        return {
            "supported": True,
            "reason": "supported",
            "wt": wt,
            "position": position,
            "mutant": mutant,
            "chain": matching[0]["chain"],
        }

    return {
        "supported": False,
        "reason": reason,
        "wt": wt,
        "position": position,
        "mutant": mutant,
        "chain": None,
    }

task_rows = []
for row in pairs.itertuples(index=False):
    info = locate_mutation(row.drug, row.mutation)
    task_rows.append({
        "drug": row.drug,
        "pdb_id": DRUG_TO_PDB[row.drug],
        "mutation": row.mutation,
        "auc_ratio_vs_wt": row.auc_ratio_vs_wt,
        "resistant": row.resistant,
        **info,
    })

tasks = pd.DataFrame(task_rows)

print("Supported pairs:", int(tasks["supported"].sum()))
print("Unsupported pairs:", int((~tasks["supported"]).sum()))
display(tasks.groupby(["drug", "supported"]).size().unstack(fill_value=0))
display(tasks.head(15))

## Step 8 — Generate the supported mutant structures

PDBFixer expects mutation notation in the form `THR-790-MET`. The code translates one-letter amino-acid notation automatically.

The ligand and other heteroatoms are retained in the output PDB. No minimization is performed yet.

In [ ]:
from pdbfixer import PDBFixer
from openmm.app import PDBFile

ONE_TO_THREE = {
    "A": "ALA", "R": "ARG", "N": "ASN", "D": "ASP", "C": "CYS",
    "E": "GLU", "Q": "GLN", "G": "GLY", "H": "HIS", "I": "ILE",
    "L": "LEU", "K": "LYS", "M": "MET", "F": "PHE", "P": "PRO",
    "S": "SER", "T": "THR", "W": "TRP", "Y": "TYR", "V": "VAL",
}

mutant_dir = Path("mutant_structures")
mutant_dir.mkdir(exist_ok=True)

def generate_mutant_structure(reference_pdb, chain, wt, position, mutant, output_pdb):
    fixer = PDBFixer(filename=str(reference_pdb))

    mutation_spec = (
        f"{ONE_TO_THREE[wt]}-{position}-{ONE_TO_THREE[mutant]}"
    )
    fixer.applyMutations([mutation_spec], chain)

    # Do not build unresolved loops. Add only atoms missing from residues
    # already present in the experimental structure.
    fixer.findMissingResidues()
    fixer.missingResidues = {}
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()

    with open(output_pdb, "w") as handle:
        PDBFile.writeFile(fixer.topology, fixer.positions, handle, keepIds=True)

    return mutation_spec

generation_rows = []

for row in tasks.itertuples(index=False):
    result = row._asdict()

    if not row.supported:
        result.update({
            "generation_status": "not_generated",
            "mutation_spec": None,
            "output_pdb": None,
            "generation_error": row.reason,
        })
        generation_rows.append(result)
        continue

    output_name = f"{row.drug}_{row.pdb_id}_{row.mutation}.pdb"
    output_path = mutant_dir / output_name

    try:
        mutation_spec = generate_mutant_structure(
            reference_pdb=reference_files[row.drug],
            chain=row.chain,
            wt=row.wt,
            position=row.position,
            mutant=row.mutant,
            output_pdb=output_path,
        )
        result.update({
            "generation_status": "generated",
            "mutation_spec": mutation_spec,
            "output_pdb": str(output_path),
            "generation_error": None,
        })
    except Exception as exc:
        result.update({
            "generation_status": "failed",
            "mutation_spec": None,
            "output_pdb": None,
            "generation_error": f"{type(exc).__name__}: {exc}",
        })

    generation_rows.append(result)

generation_report = pd.DataFrame(generation_rows)

print(generation_report["generation_status"].value_counts(dropna=False))
display(generation_report.head(15))

## Step 9 — Validate generated structures

This checks that the requested mutant residue is present in the written PDB file.

In [ ]:
def check_output_mutation(pdb_path, chain_id, position, expected_mutant):
    idx = protein_residue_index(pdb_path)
    candidates = idx.get(int(position), [])
    matches = [
        x for x in candidates
        if x["chain"] == chain_id and x["one_letter"] == expected_mutant
    ]
    return len(matches) == 1

validation = []
for row in generation_report.itertuples(index=False):
    if row.generation_status != "generated":
        validation.append(False)
        continue

    validation.append(
        check_output_mutation(
            row.output_pdb,
            row.chain,
            row.position,
            row.mutant,
        )
    )

generation_report["mutation_verified"] = validation

print(
    generation_report.groupby(
        ["generation_status", "mutation_verified"]
    ).size()
)
display(
    generation_report[
        ["drug", "mutation", "generation_status",
         "mutation_verified", "output_pdb", "generation_error"]
    ].head(20)
)

## Step 10 — Save the report and download all generated structures

The ZIP archive includes:

- all generated mutant PDB files,
- the reference PDB files,
- `mutant_generation_report.csv`, and
- `unsupported_mutations.csv`.


In [ ]:
generation_report.to_csv("mutant_generation_report.csv", index=False)

unsupported = generation_report[
    generation_report["generation_status"] != "generated"
].copy()
unsupported.to_csv("unsupported_mutations.csv", index=False)

import shutil

bundle_dir = Path("EGFR_mutant_structure_bundle")
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)

bundle_dir.mkdir()
shutil.copytree(mutant_dir, bundle_dir / "mutant_structures")
shutil.copytree(structure_dir, bundle_dir / "reference_structures")
shutil.copy("mutant_generation_report.csv", bundle_dir)
shutil.copy("unsupported_mutations.csv", bundle_dir)

archive = shutil.make_archive(
    "EGFR_mutant_structure_bundle",
    "zip",
    root_dir=bundle_dir,
)

print("Created:", archive)
files.download(archive)

## What comes next

The next notebook will calculate reproducible structural descriptors for each generated mutation–drug pair. It will begin with geometry-based features that do not require ligand force-field parameters, including:

- mutation-to-drug distance,
- local contact gain/loss,
- local packing change,
- side-chain volume and chemistry changes,
- residue solvent exposure,
- drug-pocket residue distances, and
- pocket contact fingerprints.

Ligand-aware minimization and binding-energy calculations should be added only after the covalent and noncovalent drug parameterization is validated.